In [1]:
!pip install -q kagglehub torch onnx onnxruntime scikit-learn pandas numpy joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 33.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.36.2 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.


In [2]:
import glob
import os
import random
import warnings
import joblib
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import onnx
import onnxruntime as ort
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [3]:
path = kagglehub.dataset_download("dhoogla/cicids2017")
print("Path to dataset files:", path)

data_files = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))
if not data_files:
    data_files = sorted(glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True))

if not data_files:
    raise FileNotFoundError(f"No dataset files found in {path}")

df = pd.concat([pd.read_csv(f) if f.endswith(".csv") else pd.read_parquet(f) for f in data_files], ignore_index=True)
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)

LABEL_COLUMN = "Label"
if LABEL_COLUMN not in df.columns:
    matched = [c for c in df.columns if "label" in c.lower()]
    if matched:
        LABEL_COLUMN = matched[0]
    else:
        raise ValueError("Label column not found in dataset.")

df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[LABEL_COLUMN])
df[LABEL_COLUMN] = df[LABEL_COLUMN].astype(str).str.strip()

X = df.drop(columns=[LABEL_COLUMN]).select_dtypes(include=["number"])
y = df[LABEL_COLUMN]

if X.shape[1] == 0:
    raise ValueError("No numeric features were found.")

valid_rows = X.replace([np.inf, -np.inf], np.nan).notna().all(axis=1)
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

combined = pd.concat([X, y], axis=1).drop_duplicates(ignore_index=True)
X = combined.drop(columns=[LABEL_COLUMN])
y = combined[LABEL_COLUMN]

FEATURE_COLUMNS = X.columns.tolist()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_.tolist()
NUM_CLASSES = len(class_names)
INPUT_SIZE = X.shape[1]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=SEED, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, "scaler.pkl")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print(f"Features: {INPUT_SIZE} | Classes: {NUM_CLASSES}")
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

Using Colab cache for faster access to the 'cicids2017' dataset.
Path to dataset files: /kaggle/input/cicids2017
Features: 77 | Classes: 15
Train: (1562264, 77) | Val: (334771, 77) | Test: (334771, 77)


In [4]:
class NIDSModel(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

model = NIDSModel(INPUT_SIZE, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(model, loader):
    model.eval()
    predictions, actual = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            output = model(batch_x.to(DEVICE))
            pred = torch.argmax(output, dim=1)
            predictions.extend(pred.cpu().numpy())
            actual.extend(batch_y.numpy())
    accuracy = accuracy_score(actual, predictions)
    f1 = f1_score(actual, predictions, average="macro", zero_division=0)
    return accuracy, f1

In [7]:
EPOCHS = 20
best_f1 = -1

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_accuracy, val_f1 = evaluate(model, val_loader)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_accuracy:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")

model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

test_acc, test_f1 = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_acc:.4f} | Macro F1: {test_f1:.4f}")

all_preds, all_acts = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        pred = torch.argmax(model(batch_x.to(DEVICE)), dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_acts.extend(batch_y.numpy())

print("\nClassification Report:\n", classification_report(all_acts, all_preds, target_names=class_names, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(all_acts, all_preds))

Epoch 01/20 | Loss: 0.0190 | Val Acc: 0.9947 | Val F1: 0.7180
Epoch 02/20 | Loss: 0.0187 | Val Acc: 0.9949 | Val F1: 0.6523
Epoch 03/20 | Loss: 0.0182 | Val Acc: 0.9943 | Val F1: 0.6576
Epoch 04/20 | Loss: 0.0182 | Val Acc: 0.9923 | Val F1: 0.6585
Epoch 05/20 | Loss: 0.0178 | Val Acc: 0.9950 | Val F1: 0.7310
Epoch 06/20 | Loss: 0.0176 | Val Acc: 0.9942 | Val F1: 0.7293
Epoch 07/20 | Loss: 0.0176 | Val Acc: 0.9952 | Val F1: 0.7321
Epoch 08/20 | Loss: 0.0173 | Val Acc: 0.9950 | Val F1: 0.6733
Epoch 09/20 | Loss: 0.0174 | Val Acc: 0.9950 | Val F1: 0.7254
Epoch 10/20 | Loss: 0.0172 | Val Acc: 0.9950 | Val F1: 0.6587
Epoch 11/20 | Loss: 0.0167 | Val Acc: 0.9950 | Val F1: 0.6633
Epoch 12/20 | Loss: 0.0167 | Val Acc: 0.9951 | Val F1: 0.6658
Epoch 13/20 | Loss: 0.0165 | Val Acc: 0.9946 | Val F1: 0.7333
Epoch 14/20 | Loss: 0.0166 | Val Acc: 0.9921 | Val F1: 0.7310
Epoch 15/20 | Loss: 0.0163 | Val Acc: 0.9947 | Val F1: 0.6651
Epoch 16/20 | Loss: 0.0163 | Val Acc: 0.9947 | Val F1: 0.6328
Epoch 17

In [8]:
model.eval()
dummy_input = torch.randn(1, INPUT_SIZE, dtype=torch.float32).to(DEVICE)
onnx_file = "nids_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file,
    input_names=["features"],
    output_names=["output"],
    dynamic_axes={"features": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=17,
    dynamo=False
)

onnx_model = onnx.load(onnx_file)
onnx.checker.check_model(onnx_model)

metadata = {
    "classes": "|".join(class_names),
    "features": "|".join(FEATURE_COLUMNS),
    "input_size": str(INPUT_SIZE),
    "num_classes": str(NUM_CLASSES)
}

for key, value in metadata.items():
    entry = onnx_model.metadata_props.add()
    entry.key = key
    entry.value = value

onnx.save(onnx_model, onnx_file)

session = ort.InferenceSession(onnx_file, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

sample = X_test_scaled[:10].astype(np.float32)
onnx_output = session.run([output_name], {input_name: sample})[0]
onnx_preds = np.argmax(onnx_output, axis=1)

with torch.no_grad():
    torch_output = model(torch.tensor(sample, dtype=torch.float32).to(DEVICE))
torch_preds = torch.argmax(torch_output, dim=1).cpu().numpy()

print("PyTorch Predictions:", torch_preds)
print("ONNX Predictions:   ", onnx_preds)
print("Identical:          ", np.array_equal(torch_preds, onnx_preds))

onnx_full_output = session.run([output_name], {input_name: X_test_scaled.astype(np.float32)})[0]
onnx_full_preds = np.argmax(onnx_full_output, axis=1)
print(f"\nONNX Runtime Test Acc: {accuracy_score(y_test, onnx_full_preds):.4f} | Macro F1: {f1_score(y_test, onnx_full_preds, average='macro', zero_division=0):.4f}")

try:
    from google.colab import files
    files.download("nids_model.onnx")
    files.download("scaler.pkl")
except Exception:
    pass

PyTorch Predictions: [0 2 0 0 0 0 0 0 0 0]
ONNX Predictions:    [0 2 0 0 0 0 0 0 0 0]
Identical:           True

ONNX Runtime Test Acc: 0.9949 | Macro F1: 0.7078


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>